# TP 6 — PyTorch : classer des phrases avec un réseau de neurones

Le but de ce TP est d'expérimenter avec les **MLPs** pour classifier du texte. Nous utilisons (parmi tant d'autres), le framework **PyTorch** (voir la documentation ici: https://pytorch.org/get-started/locally/) qui implémente la majorité des fonctions nécessaires pour construire et évaluer un MLP en Python.

**Objectif du TP** :

1. entraîner le classifieur du CM sur des phrases représentées par la moyenne de leurs vecteurs Word2Vec ;
2. tracer les pertes d'entraînement et de validation ;
3. optimiser les hyperparamètres ;
4. calculer l'accuracy finale sur le test, une seule fois.

**Tâche** : prédire l'auteur — Flaubert ou Zola — d'une phrase de roman.


# Récupération des données et du code précédent

Récupérez d'abord les 4 romans de la séance de révisions (deux de Flaubert, deux de Zola) et les vecteurs fastText `cc.fr.300` du TP Word2Vec qui seront utilisés comme représentation de base des mots du roman.

Ce TP utilise en outre la bibliothèque `torch` : si elle n'est pas installée, exécutez d'abord `%pip install torch` dans une cellule.


On définit ci-dessous plusieurs variables et fonctions utiles pour résoudre le TP:

- Cellule de récupération des données pour la suite de l'étude
- Cellule de chargement des vecteurs FastText
- Représentation des phrases comme la moyenne de leur embedding FastText
 

In [4]:
### RÉCUPÉRATION DES DONNÉES

import re
from collections import Counter
from pathlib import Path
from urllib.request import Request, urlopen

# Deux romans de Flaubert, deux de Zola (comme dans la séance de révisions).
ROMANS = {
    "corpus/pg14155.txt": "flaubert",   # Madame Bovary
    "corpus/pg48881.txt": "flaubert",   # Salammbô
    "corpus/pg56528.txt": "zola",       # Germinal
    "corpus/pg8600.txt":  "zola",       # L'Assommoir
}

for fichier in ROMANS:
    chemin = Path(fichier)
    if not chemin.exists():
        chemin.parent.mkdir(parents=True, exist_ok=True)
        numero = chemin.stem.removeprefix("pg")
        url = f"https://www.gutenberg.org/cache/epub/{numero}/pg{numero}.txt"
        requete = Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urlopen(requete) as reponse:
            contenu = reponse.read().decode("utf-8-sig")
        chemin.write_text(contenu, encoding="utf-8")

motif_francais = r"[^\W\d_]+(?:['’-][^\W\d_]+)*"

# Découpage en phrases (ponctuation forte, comme dans le TP Word2Vec),
# puis tokenisation ; on garde les phrases d'au moins trois tokens.
phrases = []  # liste de (tokens, auteur)
for fichier, auteur in ROMANS.items():
    texte_brut = Path(fichier).read_text(encoding="utf-8")
    texte = texte_brut.split("*** START OF", 1)[1].split("*** END OF", 1)[0]
    for brut in re.split(r"[.!?…]+", texte):
        tokens = re.findall(motif_francais, brut.lower())
        if len(tokens) >= 3:
            phrases.append((tokens, auteur))

print(f"{len(phrases):,} phrases conservées")
print(Counter(auteur for _, auteur in phrases))


31,644 phrases conservées
Counter({'zola': 19783, 'flaubert': 11861})


In [5]:
## Chargement des vecteurs FastText 

import gzip
import shutil
from pathlib import Path
from urllib.request import Request, urlopen

# Vecteurs fastText pré-entraînés (TP Word2Vec) : 1,5 Go compressé.
url = "https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.fr.300.bin.gz"
fichier_gz = Path("modele/cc.fr.300.bin.gz")
fichier_bin = Path("modele/cc.fr.300.bin")

if not fichier_bin.exists():
    fichier_gz.parent.mkdir(parents=True, exist_ok=True)
    if not fichier_gz.exists():
        print("téléchargement de", url)
        requete = Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urlopen(requete) as reponse, fichier_gz.open("wb") as sortie:
            shutil.copyfileobj(reponse, sortie)
    with gzip.open(fichier_gz, "rb") as entree, fichier_bin.open("wb") as sortie:
        shutil.copyfileobj(entree, sortie)

print("fichier prêt :", fichier_bin)


fichier prêt : modele/cc.fr.300.bin


In [6]:
from gensim.models.fasttext import load_facebook_vectors

fasttext_fr = load_facebook_vectors("modele/cc.fr.300.bin")

print("dimension :", fasttext_fr.vector_size)
print("types :", len(fasttext_fr))

dimension : 300
types : 2000000


In [15]:
## Moyenne pour représenter une phrase
# => On choisit de moyenner les embeddings pour chacun des mots de la phrase

import numpy as np

# Vecteur d'une phrase = moyenne des vecteurs fastText de ses tokens
# (les tokens hors vocabulaire sont ignorés).
def vecteur_phrase(tokens):
    vecteurs = [fasttext_fr[t] for t in tokens if t in fasttext_fr]
    if not vecteurs:
        return None
    return np.mean(vecteurs, axis=0)

auteurs = sorted({auteur for _, auteur in phrases})
auteur_vers_id = {auteur: i for i, auteur in enumerate(auteurs)}
print("classes :", auteur_vers_id)

X, y = [], []
for tokens, auteur in phrases:
    v = vecteur_phrase(tokens)
    if v is not None:  # on écarte les phrases sans aucun mot connu
        X.append(v)
        y.append(auteur_vers_id[auteur])
X = np.asarray(X, dtype=np.float32)
y = np.asarray(y, dtype=np.int64)
print("matrice X :", X.shape, " — étiquettes :", dict(zip(auteurs, np.bincount(y))))

# Découpage train / validation / test (70 / 15 / 15), stratifié par auteur.
rng = np.random.default_rng(42)

def decouper(indices):
    rng.shuffle(indices)
    n = len(indices)
    n_train, n_val = int(0.70 * n), int(0.15 * n)
    return indices[:n_train], indices[n_train:n_train + n_val], indices[n_train + n_val:]

train_idx, val_idx, test_idx = [], [], []
for classe in np.unique(y):
    a, b, c = decouper(np.flatnonzero(y == classe))
    train_idx.append(a)
    val_idx.append(b)
    test_idx.append(c)
train_idx = np.concatenate(train_idx)
val_idx = np.concatenate(val_idx)
test_idx = np.concatenate(test_idx)

print(f"train : {len(train_idx):,}  val : {len(val_idx):,}  test : {len(test_idx):,}")

classes : {'flaubert': 0, 'zola': 1}
matrice X : (31644, 300)  — étiquettes : {'flaubert': np.int64(11861), 'zola': np.int64(19783)}
train : 22,150  val : 4,746  test : 4,748


### Exercice 0:
Commenter chacune des instructions des fonctions ci-dessous, en écrivant à la fois les commentaires ligne par ligne et les docstrings.

In [9]:
## Fonctions de base

import torch
from torch import nn, optim
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)

d = X.shape[1]          # dimension des vecteurs de phrases (300)
nb_classes = len(auteurs)

def fabrique_loader(X_part, y_part, batch_size, melanger):
    dataset = TensorDataset(torch.from_numpy(X_part), torch.from_numpy(y_part))
    return DataLoader(dataset, batch_size=batch_size, shuffle=melanger)

@torch.no_grad()
def evaluer(modele, loader, perte_fn=None):
    # Renvoie (perte moyenne, accuracy) sur un loade.
    if perte_fn is None:
        perte_fn = nn.CrossEntropyLoss()
    modele.eval()
    somme_perte, justes, n = 0.0, 0, 0
    for xb, yb in loader:
        sortie = modele(xb)
        somme_perte += perte_fn(sortie, yb).item() * len(yb)
        justes += (sortie.argmax(dim=1) == yb).sum().item()
        n += len(yb)
    return somme_perte / n, justes / n

def entrainer(modele, loader_train, loader_val, epochs, lr, weight_decay=0.0, verbose=True):
    # Boucle d'entraînement.
    perte_fn = nn.CrossEntropyLoss()
    optimiseur = optim.SGD(modele.parameters(), lr=lr, weight_decay=weight_decay)
    historique = {"perte_train": [], "perte_val": [], "acc_val": []}
    for époque in range(epochs):
        modele.train()
        somme, n = 0.0, 0
        for xb, yb in loader_train:
            optimiseur.zero_grad()
            perte = perte_fn(modele(xb), yb)
            perte.backward()
            optimiseur.step()
            somme += perte.item() * len(yb)
            n += len(yb)
        historique["perte_train"].append(somme / n)
        perte_val, acc_val = evaluer(modele, loader_val, perte_fn)
        historique["perte_val"].append(perte_val)
        historique["acc_val"].append(acc_val)
        if verbose and (époque % 5 == 0 or époque == epochs - 1):
            print(f"époque {époque + 1:>3}/{epochs}  perte_train={historique['perte_train'][-1]:.4f}"
                  f"  perte_val={perte_val:.4f}  acc_val={acc_val:.3f}")
    return historique


# Partie 1 — Le classifieur du CM en PyTorch

# Exercice 1 — Entraîner le classifieur linéaire (softmax)

Le classifieur du CM est une simple couche linéaire $W \in \mathbb{R}^{2 \times d}$ + biais, suivie du softmax (dans PyTorch, `nn.CrossEntropyLoss` applique déjà le softmax avant l'entropie croisée).

**Questions:**
1. Définissez `classifieur = nn.Linear(d, nb_classes)` et entraînez-le avec la fonction `entrainer` (30 époques, `lr = 0.1`, batchs de 256). Combien de paramètres ce modèle contient-il ?
2. Quelle accuracy de validation obtenez-vous à la fin ?


In [12]:
## Chargement des données au format PyTorch
batch_size = 256
loader_train = fabrique_loader(X[train_idx], y[train_idx], batch_size, melanger=True)
loader_val = fabrique_loader(X[val_idx], y[val_idx], batch_size, melanger=False)
loader_test = fabrique_loader(X[test_idx], y[test_idx], batch_size, melanger=False)


## À remplir

# Exercice 2 — Tracer les pertes d'entraînement et de validation

**Questions**:
- Tracez sur un même graphique la perte d'entraînement et la perte de validation au fil des époques (stockées dans la cellule précédente dans la variable `historique_lineaire`). 
- Le modèle sous-apprend-il, ou commence-t-il à sur-apprendre ?
- Changez les paramètres de votre entraînement pour forcer le modèle à faire du sur-apprentissage.

# Partie 2 — Perceptron multi-couche (MLP)

# Exercice 3 — Ajouter une couche cachée

Le MLP du CM intercale une couche cachée de taille $h$ avec une activation ReLU entre l'entrée et la sortie :

$$\mathbf{h} = \max(0,\, W_1 \mathbf{x} + \mathbf{b}_1), \qquad \hat{y} = \mathrm{softmax}(W_2 \mathbf{h} + \mathbf{b}_2)$$

**Questions**:
- Commentez en détail le code ci-dessous qui permet de définir la classe `MLP` qui hérite de `nn.Module`.
- Entraînez-le avec la même fonction `entrainer`, puis tracez ses pertes à côté de celles du classifieur linéaire. 
- Le MLP améliore-t-il l'accuracy de validation ?


In [14]:
class MLP(nn.Module):
    def __init__(self, d, h, nb_classes):
        super().__init__()
        self.couche_cachee = nn.Linear(d, h)
        self.activation = nn.ReLU()
        self.sortie = nn.Linear(h, nb_classes)

    def forward(self, x):
        return self.sortie(self.activation(self.couche_cachee(x)))

torch.manual_seed(42)
mlp = MLP(d, h=64, nb_classes=nb_classes)
print("paramètres :", sum(p.numel() for p in mlp.parameters()))


paramètres : 19394


# Partie 3 — Optimisation des hyperparamètres

# Exercice 4 — Recherche sur grille (grid search)
**Questions**:
- Faites varier les hyperparamètres du MLP — `lr` $\in \{0.01,\, 0.1,\, 0.5\}$, `h` $\in \{32,\, 64,\, 128\}$ et `weight_decay` $\in \{0,\, 10^{-4}\}$ — à nombre d'époques (30) et taille de batch (256) fixés. Pour chaque configuration, entraînez un MLP (graine fixée à 42).
- Relevez sa perte et son accuracy de validation finales, puis choisissez la meilleure configuration pour votre jeu de données.


# Partie 4 — Évaluation finale sur le test

# Exercice 5 — Accuracy finale

**Rappel méthodologique** : le jeu de **test** ne doit servir qu'une seule fois, à la toute fin, pour évaluer le modèle choisi grâce à la validation. **On ne peut pas entraîner ni choisir le modèle sur le test**.

**Questions:**
- Évaluez le meilleur modèle de l'exercice 4 sur le jeu de test et rapportez son accuracy (et sa perte).
- Comparez avec son accuracy de validation : la tâche Flaubert/Zola est-elle bien apprise en utilisant en entrée la moyenne des vecteurs Word2Vec ?

## Pour aller plus loin

- Ajoutez une deuxième couche cachée, testez d'autres activations (tanh, GELU) ou du dropout afin de vérifier si celles-ci permettent de monter l'activation.
- Remplacez `optim.SGD` par `optim.Adam`, ou ajoutez un scheduler de learning rate.

